# VAR 2026 Colab runner — scene BTS thật (dữ liệu thi)

Notebook này dùng để chạy **dữ liệu thi thật** (`HCM0421`, `HCM0539`, `HCM0540`, `HCM0644`, `HCM0674` — camera SIMPLE_RADIAL, đi qua bước undistort ảnh méo).

**Quan trọng — cả 7 scene đều bắt buộc phải nộp**, không chỉ 5 scene ở đây: `bonsai`/`chair` (chạy ở `notebooks/colab_runner_bonsai.ipynb`) cũng phải có mặt trong `submission.zip` cuối cùng. Sau khi cả 2 notebook đã chạy xong, chạy nốt **Bước 10** ở cuối file này để ghép cả 7 scene thành 1 `submission.zip` duy nhất — 2 notebook chạy riêng sẽ ghi đè `submission.zip` của nhau nếu không ghép lại.

Chạy từng cell theo thứ tự từ trên xuống (Runtime > Run all cũng được, nhưng lần đầu nên chạy tay từng cell để dễ phát hiện lỗi sớm).

**Trước khi chạy — chỉ cần chuẩn bị đúng 2 thứ:**
1. **Notebook này**: không cần tự upload file — mở trực tiếp trên Colab qua **File > Open notebook > tab GitHub**, dán URL repo (`https://github.com/viethoang2503/viettel-ai-race-bts-digital`), chọn `notebooks/colab_runner_hcm.ipynb`.
2. **Dataset**: upload toàn bộ thư mục `VAI_NVS_DATA_ROUND2/` lên Google Drive, đặt đúng đường dẫn `MyDrive/var2026/VAI_NVS_DATA_ROUND2/` (Bước 3 bên dưới sẽ tự link nó vào code qua symlink — không copy lại, không tốn thêm dung lượng).

**An toàn khi bị ngắt kết nối:** notebook này chạy lại được từ đầu bất cứ lúc nào mà không mất tiến độ — cell clone tự `git pull` để lấy code mới nhất nếu code đã có sẵn, và `real_train_fn` tự bỏ qua scene nào đã train xong (dựa trên checkpoint + fingerprint dữ liệu đã lưu trên Drive, lưu định kỳ mỗi 5000 iteration chứ không chỉ ở cuối).

## Bước 0 — Kiểm tra GPU runtime
Phải thấy tên GPU (vd Tesla T4/A100/L4) ở dưới. Nếu báo lỗi "command not found" hoặc không thấy GPU nào: vào menu **Runtime > Change runtime type > Hardware accelerator > GPU**, chọn Save, rồi chạy lại cell này.

In [ ]:
!nvidia-smi

## Bước 1 — Clone code từ GitHub
Chỉ clone nếu chưa có; nếu đã có sẵn (chạy lại notebook trong cùng session Colab) thì tự `git pull` để luôn lấy code mới nhất — quan trọng nếu có fix mới được push lên trong lúc bạn đang test.

**Quan trọng — `git pull` là CHƯA ĐỦ nếu bạn đã chạy pipeline (Bước 5) ít nhất 1 lần trong session này:** Python cache code Python đã import vào bộ nhớ của kernel; `git pull` chỉ cập nhật file trên đĩa chứ không nạp lại code đang chạy. Nếu bạn gặp lỗi, đã `git pull`, mà chạy lại Bước 5 vẫn lỗi y hệt như cũ — vào **Runtime > Restart session** rồi chạy lại từ Bước 1. Không cần build lại CUDA extension hay tải lại LPIPS model, mọi thứ vẫn nằm trên đĩa/Drive, restart chỉ reset tiến trình Python.

In [ ]:
import os

REPO_URL = "https://github.com/viethoang2503/viettel-ai-race-bts-digital.git"
REPO_DIR = "/content/viettel-ai-race-bts-digital"

if not os.path.isdir(REPO_DIR):
    !git clone --recurse-submodules {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git submodule update --init --recursive

## Bước 2 — Mount Google Drive
Phải chạy trong MỘT cell Python riêng như thế này (không được gọi qua `!bash ...`) — `drive.mount()` cần chạy trực tiếp trong kernel của notebook mới hiện được popup xin quyền, gọi qua subprocess của lệnh `!bash` sẽ lỗi `AttributeError: 'NoneType' object has no attribute 'kernel'`.

Sẽ hiện popup xin quyền truy cập Drive — bấm cho phép.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Bước 3 — Cài môi trường + link dataset
Lần chạy đầu tiên sẽ build 2 CUDA extension từ source (**5-10 phút**, thấy dòng "Building ... from source"); từ lần thứ 2 trở đi (kể cả sau khi bị ngắt kết nối, mở lại) sẽ restore từ cache trên Drive và chạy trong vài giây (dòng "Restoring cached ..."). Cache này dùng chung với notebook bonsai — nếu đã chạy notebook đó trước, bước này sẽ nhanh ngay từ lần đầu.

Cell này cũng tự tạo symlink `VAI_NVS_DATA_ROUND2 -> MyDrive/var2026/VAI_NVS_DATA_ROUND2` — nếu thấy dòng `WARNING: dataset not found`, nghĩa là bạn chưa upload dataset lên đúng đường dẫn đó trên Drive; upload xong thì chạy lại cell này. Nếu thấy `ERROR: Google Drive is not mounted`, quay lại Bước 2 chạy cell mount trước.

**Kiểm tra:** dòng cuối cùng của output phải là `CUDA available: True`. Nếu là `False`, quay lại Bước 0 kiểm tra runtime type.

In [ ]:
!bash environment/setup_colab.sh

## Bước 4 — Chọn scene BTS thật
`RUN_MODE` nhận:
- Tên 1 scene cụ thể (mặc định `"HCM0421"`) — khuyến nghị chạy từng scene riêng lẻ trước để chắc chắn undistort + train + render đều ổn.
- `"all_hcm"` — chạy cả 5 scene BTS thật cùng lúc (chỉ nên dùng sau khi ít nhất 1 scene riêng lẻ đã chạy ổn).

5 scene BTS thật: `HCM0421`, `HCM0539`, `HCM0540`, `HCM0644`, `HCM0674`.

In [ ]:
RUN_MODE = "HCM0421"  # tên 1 scene BTS (vd "HCM0421") | "all_hcm"
ITERATIONS = 30000

HCM_SCENE_NAMES = {"HCM0421", "HCM0539", "HCM0540", "HCM0644", "HCM0674"}
assert RUN_MODE == "all_hcm" or RUN_MODE in HCM_SCENE_NAMES, f"RUN_MODE không hợp lệ: {RUN_MODE!r}"

## Bước 5 — Chạy pipeline
Với mỗi scene: undistort ảnh méo SIMPLE_RADIAL -> PINHOLE -> train 2 lần (eval + full data) -> render holdout để tính điểm -> render `test_poses.csv` thật -> đóng gói `submission.zip`. Toàn bộ output (checkpoint, log, ảnh render, zip) ghi thẳng ra Drive tại `OUTPUT_ROOT` bên dưới — không mất gì nếu Colab bị ngắt giữa chừng, chỉ cần chạy lại notebook từ đầu (checkpoint lưu mỗi 5000 iteration, không chỉ ở cuối, nên tối đa chỉ mất gần 5000 iteration nếu bị ngắt).

In [ ]:
import functools
from pathlib import Path

from src.common.config import load_scenes
from src.evaluation.compute_metrics import load_lpips_model
from src.orchestrator.run_pipeline import run_baseline_pipeline
from src.rendering.gs_render_fn import real_render_fn
from src.training.gs_train_fn import real_train_fn

OUTPUT_ROOT = Path("/content/drive/MyDrive/var2026/outputs")

all_scenes = load_scenes()
if RUN_MODE == "all_hcm":
    scenes = [s for s in all_scenes if s.name in HCM_SCENE_NAMES]
else:
    scenes = [s for s in all_scenes if s.name == RUN_MODE]
print(f"RUN_MODE={RUN_MODE} -> chạy {len(scenes)} scene: {[s.name for s in scenes]}")

train_fn = functools.partial(real_train_fn, iterations=ITERATIONS)

result = run_baseline_pipeline(
    scenes=scenes,
    train_fn=train_fn,
    render_fn=real_render_fn,
    lpips_model=load_lpips_model(),
    psnr_max=30.0,
    output_root=OUTPUT_ROOT,
)

## Bước 6 — Đọc kết quả
- `Per-scene scores`: điểm holdout tự đo (0-1, càng cao càng tốt), kèm breakdown LPIPS/SSIM/PSNR riêng.
- `Skipped scenes`: scene bị bỏ qua do lỗi data — nếu có, `submission_zip` sẽ luôn là `None` (fail-closed, cả bài nộp bị giữ lại chứ không nộp thiếu scene).
- `Validation problems`: submission.zip đã đóng gói nhưng có lỗi (sai kích thước ảnh, thiếu file...) — zip vẫn nằm trên Drive để debug nhưng không được coi là hợp lệ.
- `submission_zip`: đường dẫn file `submission.zip` trên Drive nếu mọi thứ hợp lệ — đây là file để nộp.

In [ ]:
print("Per-scene scores:")
for name, score in result.per_scene_scores.items():
    metrics = result.per_scene_metrics.get(name, {})
    detail = ", ".join(f"{k}={v:.4f}" for k, v in metrics.items())
    print(f"  {name}: {score:.4f}  ({detail})")

if result.skipped_scenes:
    print("\nSkipped scenes:")
    for name, problems in result.skipped_scenes.items():
        print(f"  {name}: {problems}")

if result.validation_problems:
    print("\nValidation problems (submission withheld):")
    for problem in result.validation_problems:
        print(f"  {problem}")

print(f"\nsubmission_zip: {result.submission_zip}")
print(f"Toàn bộ output (checkpoint/log/render) nằm tại: {OUTPUT_ROOT}")

## Bước 7 — Kiểm tra mắt thường trước khi nộp (bắt buộc, không phải tuỳ chọn)
Điểm số tự động trên holdout không chắc phản ánh đúng chất lượng render trên `test_poses.csv` thật (test pose có thể extrapolate xa hơn holdout tự tạo — xa đến mức model không có Gaussian nào phủ tới, render ra gần như trắng trơn). `validate_submission` chỉ kiểm tra đúng kích thước/định dạng ảnh, **không kiểm tra nội dung ảnh có hợp lý hay không** — một ảnh trắng đúng kích thước vẫn PASS hết mọi kiểm tra tự động và mất điểm oan khi BTC chấm.

Cell dưới quét **toàn bộ** ảnh `test_render` của từng scene (không chỉ vài ảnh mẫu), tự động gắn cờ ảnh nghi ngờ bị trắng/trống dựa trên độ lệch chuẩn pixel, rồi hiển thị: vài ảnh mẫu ngẫu nhiên + toàn bộ ảnh bị gắn cờ (nếu có). **Chỉ xem, không sửa ảnh** (sửa thủ công ảnh đầu ra bị cấm theo đề bài) — nếu thấy ảnh trắng/hỏng thật, cần cải thiện model (train thêm, thêm kỹ thuật ở Plan 2) chứ không phải sửa tay ảnh output.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

from src.submission.visual_qa import find_suspicious_renders


def _show_images(paths, title_prefix):
    if not paths:
        return
    fig, axes = plt.subplots(1, len(paths), figsize=(5 * len(paths), 5))
    axes = [axes] if len(paths) == 1 else axes
    for ax, path in zip(axes, paths):
        ax.imshow(Image.open(path))
        ax.set_title(f"{title_prefix}: {path.name}")
        ax.axis("off")
    plt.show()


for scene in scenes:
    render_dir = OUTPUT_ROOT / scene.name / "test_render"
    if not render_dir.exists():
        continue

    all_renders = sorted(p for p in render_dir.iterdir() if p.is_file())
    suspicious = find_suspicious_renders(render_dir)

    print(f"== {scene.name}: {len(all_renders)} ảnh, {len(suspicious)} ảnh bị nghi ngờ trắng/trống ==")
    for path, std in suspicious:
        print(f"  NGHI NGỜ: {path.name}  (pixel std={std:.2f})")

    _show_images(all_renders[:3], f"{scene.name} — mẫu ngẫu nhiên")
    _show_images([p for p, _ in suspicious], f"{scene.name} — BỊ NGHI NGỜ")

## Bước 8 — Chẩn đoán holdout trước khi đầu tư thêm GPU (Giai đoạn 0)

Không tốn GPU — chỉ đọc lại ảnh holdout đã render sẵn từ Bước 5/6 và so trực tiếp với ảnh gốc (ground-truth) tương ứng, khác với Bước 7 (không có ground-truth cho `test_poses.csv` thật). Với mỗi scene, hiển thị 5 ảnh holdout tệ nhất (predicted cạnh ground-truth) kèm LPIPS/SSIM/PSNR từng ảnh — dùng để quyết định `bonsai`/`chair` cần thử hyperparameter nào ở giai đoạn sau.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

from src.common.config import load_scenes
from src.diagnostics.scene_diagnosis import compute_per_image_metrics, rank_holdout_by_score
from src.evaluation.compute_metrics import load_lpips_model
from src.training.undistort_scene import undistort_scene

lpips_model = load_lpips_model()
all_7_scenes = load_scenes()

for scene in all_7_scenes:
    scene_output = OUTPUT_ROOT / scene.name
    pred_dir = scene_output / "holdout_render"
    if not pred_dir.exists():
        print(f"== {scene.name}: chưa có holdout_render, bỏ qua ==")
        continue

    working_scene = undistort_scene(scene, scene_output / "undistorted")
    per_image = compute_per_image_metrics(pred_dir, working_scene.train_images_dir, lpips_model, psnr_max=30.0)
    worst = rank_holdout_by_score(per_image)[:5]
    if not worst:
        print(f"== {scene.name}: không có ảnh holdout khớp ground-truth, bỏ qua ==")
        continue

    print(f"== {scene.name}: {len(per_image)} ảnh holdout, 5 ảnh tệ nhất ==")
    for name, score in worst:
        m = per_image[name]
        print(f"  {name}: score={score:.4f}  lpips={m['lpips']:.4f} ssim={m['ssim']:.4f} psnr={m['psnr']:.2f}")

    fig, axes = plt.subplots(2, len(worst), figsize=(5 * len(worst), 10))
    for i, (name, score) in enumerate(worst):
        axes[0, i].imshow(Image.open(pred_dir / name).convert("RGB"))
        axes[0, i].set_title(f"predicted: {name}")
        axes[0, i].axis("off")
        axes[1, i].imshow(Image.open(working_scene.train_images_dir / name).convert("RGB"))
        axes[1, i].set_title(f"ground-truth: {name}")
        axes[1, i].axis("off")
    plt.suptitle(scene.name)
    plt.show()

## Bước 9 — Giai đoạn 1: ma trận 5 biến thể cho từng scene HCM

Mỗi scene chạy riêng 1 lệnh (không gộp cả 5 scene vào 1 lệnh) — nếu Colab bị ngắt giữa chừng chỉ mất tiến độ của đúng 1 scene, không mất cả 5. Mỗi variant lưu checkpoint mỗi 5000 iteration và tự resume đúng seed/config khi chạy lại. Screening ở 15000 iteration; biến thể thắng (hoặc sít sao, tự động re-run) được train lại ở 30000 iteration đầy đủ trước khi ship.

In [ ]:
import functools

from src.orchestrator.run_experiment_matrix import run_experiment_matrix_pipeline
from src.postprocess.prune_floaters import prune_checkpoint
from src.submission.reproducibility_bundle import write_reproducibility_bundle
from src.training.train_variant import ALL_TRAINING_VARIANTS, run_training_variant

screening_train_fn = functools.partial(run_training_variant, iterations=15000)
final_train_fn = functools.partial(run_training_variant, iterations=30000)
matrix_lpips_model = load_lpips_model()

# Chỉ test 3/5 biến thể để tiết kiệm GPU cho Giai đoạn 2/3 (baseline luôn
# bắt buộc — là mốc so sánh; depth_reg là kỹ thuật ROI cao nhất cho dữ liệu
# BTS theo phân tích debai.md; full_stack đại diện "bật hết" antialiasing +
# appearance_embed cộng dồn với depth_reg, không cần test riêng từng cái).
# Đổi lại thành ALL_TRAINING_VARIANTS nếu muốn so sánh đầy đủ cả 5.
REDUCED_VARIANTS = [
    v for v in ALL_TRAINING_VARIANTS if v.name in ("baseline", "depth_reg", "full_stack")
]

hcm_scenes = [s for s in load_scenes() if s.name.startswith("HCM")]
matrix_results = {}
for scene in hcm_scenes:
    result = run_experiment_matrix_pipeline(
        scenes=[scene],
        screening_train_fn=screening_train_fn,
        final_train_fn=final_train_fn,
        render_fn=real_render_fn,
        prune_fn=prune_checkpoint,
        lpips_model=matrix_lpips_model,
        psnr_max=30.0,
        vram_budget_bytes=20 * 1024**3,
        output_root=OUTPUT_ROOT,
        variants=REDUCED_VARIANTS,
    )
    matrix_results[scene.name] = result
    if scene.name not in result.chosen_config:
        print(f"{scene.name}: pipeline không tạo được candidate hợp lệ: {result.validation_problems}")
        continue
    write_reproducibility_bundle(
        scene.name, result.chosen_config[scene.name], result.all_candidates[scene.name],
        OUTPUT_ROOT / "reproducibility",
    )
    print(f"{scene.name}: winner={result.chosen_config[scene.name]['variant']} "
          f"floater_cleanup={result.chosen_config[scene.name]['floater_cleanup']} "
          f"score={result.per_scene_scores[scene.name]:.4f}")

## Bước 10 — Ghép cả 7 scene thành 1 `submission.zip` cuối cùng (bắt buộc trước khi nộp)

Chỉ chạy cell này **sau khi cả `colab_runner_hcm.ipynb` (5 scene BTS) VÀ `colab_runner_bonsai.ipynb` (`bonsai`+`chair`) đều đã chạy xong** — mỗi notebook train/render độc lập rồi ghi checkpoint/ảnh ra Drive, nhưng mỗi lần gọi pipeline lại tự đóng gói `submission.zip` riêng (ghi đè cái trước) chỉ với scene của lần chạy đó. Cell này không train/render gì thêm — chỉ **quét lại** toàn bộ ảnh `test_render` đã có sẵn trên Drive cho **cả 7 scene**, đóng gói lại thành 1 file `submission.zip` duy nhất, rồi validate đầy đủ.

Có thể chạy cell này bất cứ lúc nào, từ notebook nào cũng được (không phụ thuộc biến `scenes`/`result` của lần chạy Bước 5 phía trên) — miễn Drive đã có đủ `test_render` cho cả 7 scene.

In [ ]:
from pathlib import Path

from src.common.config import load_scenes
from src.submission.package_submission import package_submission
from src.submission.validate_submission import validate_submission
from src.submission.visual_qa import find_suspicious_renders

OUTPUT_ROOT = Path("/content/drive/MyDrive/var2026/outputs")

all_7_scenes = load_scenes()  # always all 7, from configs/scenes.yaml
final_zip = OUTPUT_ROOT / "submission.zip"

scene_render_dirs = {}
missing = []
for scene in all_7_scenes:
    render_dir = OUTPUT_ROOT / scene.name / "test_render"
    if not render_dir.exists() or not any(render_dir.iterdir()):
        missing.append(scene.name)
        continue
    scene_render_dirs[scene.effective_submission_dir] = render_dir

if missing:
    print(f"CẢNH BÁO: {len(missing)} scene CHƯA có ảnh test_render, sẽ KHÔNG ghép vào: {missing}")
    print("Chạy notebook tương ứng cho (các) scene này trước rồi chạy lại cell này.\n")

package_submission(scene_render_dirs, final_zip)
problems = validate_submission(final_zip, all_7_scenes)

print(f"submission.zip cuối cùng: {final_zip}")
print(f"Gồm {len(scene_render_dirs)}/7 scene: {sorted(scene_render_dirs.keys())}")

if problems:
    print(f"\n{len(problems)} VẤN ĐỀ — submission CHƯA hợp lệ, không nộp file này:")
    for p in problems:
        print(f"  {p}")
else:
    print("\nHỢP LỆ — sẵn sàng nộp.")

print("\n== Quét ảnh nghi ngờ trắng/trống (cả 7 scene) ==")
total_suspicious = 0
for name, render_dir in scene_render_dirs.items():
    suspicious = find_suspicious_renders(render_dir)
    total_suspicious += len(suspicious)
    if suspicious:
        print(f"  {name}: {len(suspicious)} ảnh nghi ngờ — {[p.name for p, _ in suspicious]}")
print(f"Tổng: {total_suspicious} ảnh nghi ngờ trên toàn bộ submission.")

## Bước 11 — Đóng gói reproducibility bundle (riêng, KHÔNG nằm trong submission.zip)

`submission.zip` chỉ được chứa ảnh render theo đúng định dạng đề bài — mọi config/log/score table nộp kèm (nếu BTC yêu cầu) phải nằm trong 1 file zip riêng.

In [ ]:
from src.submission.reproducibility_bundle import package_reproducibility_bundle

reproducibility_dir = OUTPUT_ROOT / "reproducibility"
reproducibility_zip = OUTPUT_ROOT / "reproducibility_bundle.zip"
expected_scene_names = [scene.name for scene in load_scenes()]
try:
    package_reproducibility_bundle(
        reproducibility_dir, reproducibility_zip, expected_scene_names,
    )
    print(f"reproducibility_bundle.zip: {reproducibility_zip}")
except ValueError as error:
    print(f"WARNING: chưa thể đóng bundle -- {error}")